# 02 - Training: Base Case, School Baseline, and Karatsuba

This notebook trains three models:
1. **4-bit base case** — verify the model can learn small multiplication perfectly
2. **8-bit school baseline** — establish baseline with school algorithm traces
3. **8-bit Karatsuba** — main experiment with recursive decomposition

**Run on Colab with T4 GPU (free tier) or TPU v2/v3.**

Checkpoints saved to Google Drive. Metrics logged to Weights & Biases.

In [ ]:
# Cell 1: Setup — install dependencies, mount Drive, init wandb
import os
import sys

# --- Clone and install repo ---
REPO_URL = "https://github.com/YOUR_USERNAME/karatsuba-transformers.git"  # TODO: update
REPO_DIR = "/content/karatsuba-transformers"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

!pip install -q -r {REPO_DIR}/requirements.txt
!pip install -q -e {REPO_DIR}

sys.path.insert(0, REPO_DIR)

# --- Mount Google Drive for checkpoints ---
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_BASE = '/content/drive/MyDrive/karatsuba_checkpoints'
os.makedirs(CHECKPOINT_BASE, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_BASE}")

# --- Initialize Weights & Biases ---
import wandb
wandb.login()  # Will prompt for API key on first run

# --- Detect hardware ---
import jax
print(f"JAX version: {jax.__version__}")
print(f"Available devices: {jax.devices()}")
print(f"Device type: {jax.devices()[0].device_kind}")

# Use bfloat16 on TPU, float32 on GPU (T4 doesn't support bf16 well)
device_kind = jax.devices()[0].device_kind
if 'TPU' in device_kind:
    print("Running on TPU — using bfloat16 precision")
    PRECISION = 'bfloat16'
elif 'GPU' in device_kind or 'NVIDIA' in device_kind.upper():
    print("Running on GPU — using float32 precision")
    PRECISION = 'float32'
else:
    print("Running on CPU — using float32 precision")
    PRECISION = 'float32'

import yaml
import numpy as np
import matplotlib.pyplot as plt

print("\nSetup complete.")

In [ ]:
# Cell 2: Train 4-bit base case
# Phase 1: The model must learn to multiply all 4-bit pairs perfectly.
# This is the foundation — if this fails, nothing else will work.

from src.training.train import Trainer
from src.data.dataset import MultiplicationDataset
from src.model.looped_transformer import KaratsubaTransformer

# Load config
config_path = os.path.join(REPO_DIR, 'configs', '4bit_base_case.yaml')
with open(config_path, 'r') as f:
    config_4bit = yaml.safe_load(f)

# Override checkpoint dir and precision
config_4bit['training']['checkpoint_dir'] = os.path.join(CHECKPOINT_BASE, '4bit_base_case')
os.makedirs(config_4bit['training']['checkpoint_dir'], exist_ok=True)

print("=" * 60)
print("Phase 1: Training 4-bit base case")
print("=" * 60)
print(f"Model: d={config_4bit['model']['d_model']}, "
      f"heads={config_4bit['model']['n_heads']}, "
      f"layers={config_4bit['model']['n_shared_layers']}")
print(f"Loops: {config_4bit['model']['loop']['n_loops']}")
print(f"Training steps: {config_4bit['training']['n_steps']}")
print(f"Batch size: {config_4bit['training']['batch_size']}")
print()

# Initialize wandb run
wandb.init(
    project=config_4bit['experiment']['wandb_project'],
    name=config_4bit['experiment']['name'],
    tags=config_4bit['experiment']['wandb_tags'],
    config=config_4bit,
)

# Build dataset
dataset_4bit = MultiplicationDataset(config_4bit['data'])
print(f"Dataset size: {len(dataset_4bit)} examples")

# Build model
import jax.random as jrandom
key = jrandom.PRNGKey(config_4bit['experiment']['seed'])
model_4bit = KaratsubaTransformer(config_4bit['model'], key=key)

# Count parameters
import jax.tree_util as jtu
n_params = sum(x.size for x in jtu.tree_leaves(model_4bit) if hasattr(x, 'size'))
print(f"Model parameters: {n_params:,}")

# Train
trainer_4bit = Trainer(config_4bit['training'], model_4bit, dataset_4bit)
model_4bit, history_4bit = trainer_4bit.train()

# Evaluate
final_acc = history_4bit['eval_exact_match'][-1]
print(f"\nFinal exact-match accuracy: {final_acc:.4f}")
if final_acc >= 0.99:
    print("SUCCESS: Base case learned perfectly.")
else:
    print("WARNING: Base case not perfect. Debug before proceeding.")

wandb.finish()

In [ ]:
# Cell 3: Train 8-bit school algorithm baseline
# Phase 2: Establish a baseline with the school (long) multiplication algorithm.
# Same architecture, same training — only the trace format differs.

config_path = os.path.join(REPO_DIR, 'configs', '8bit_school_baseline.yaml')
with open(config_path, 'r') as f:
    config_school = yaml.safe_load(f)

config_school['training']['checkpoint_dir'] = os.path.join(CHECKPOINT_BASE, '8bit_school_baseline')
os.makedirs(config_school['training']['checkpoint_dir'], exist_ok=True)

print("=" * 60)
print("Phase 2: Training 8-bit school algorithm baseline")
print("=" * 60)
print(f"Model: d={config_school['model']['d_model']}, "
      f"heads={config_school['model']['n_heads']}, "
      f"layers={config_school['model']['n_shared_layers']}")
print(f"Algorithm: school (long multiplication)")
print(f"Training steps: {config_school['training']['n_steps']}")
print()

wandb.init(
    project=config_school['experiment']['wandb_project'],
    name=config_school['experiment']['name'],
    tags=config_school['experiment']['wandb_tags'],
    config=config_school,
)

# Build dataset with school algorithm traces
dataset_school = MultiplicationDataset(config_school['data'])
print(f"Dataset size: {len(dataset_school)} examples")

# Build model (same architecture as Karatsuba)
key = jrandom.PRNGKey(config_school['experiment']['seed'])
model_school = KaratsubaTransformer(config_school['model'], key=key)
n_params = sum(x.size for x in jtu.tree_leaves(model_school) if hasattr(x, 'size'))
print(f"Model parameters: {n_params:,}")

# Train
trainer_school = Trainer(config_school['training'], model_school, dataset_school)

try:
    model_school, history_school = trainer_school.train()
except KeyboardInterrupt:
    print("\nTraining interrupted. Saving checkpoint...")
    trainer_school.save_checkpoint()
    history_school = trainer_school.get_history()

print(f"\nFinal 8-bit exact-match accuracy: {history_school['eval_exact_match'][-1]:.4f}")
wandb.finish()

In [ ]:
# Cell 4: Train 8-bit Karatsuba
# Phase 3: The main experiment. Train with Karatsuba-decomposed traces.
# This is where length generalisation should emerge.

config_path = os.path.join(REPO_DIR, 'configs', '8bit_karatsuba.yaml')
with open(config_path, 'r') as f:
    config_karatsuba = yaml.safe_load(f)

config_karatsuba['training']['checkpoint_dir'] = os.path.join(CHECKPOINT_BASE, '8bit_karatsuba')
os.makedirs(config_karatsuba['training']['checkpoint_dir'], exist_ok=True)

print("=" * 60)
print("Phase 3: Training 8-bit Karatsuba")
print("=" * 60)
print(f"Model: d={config_karatsuba['model']['d_model']}, "
      f"heads={config_karatsuba['model']['n_heads']}, "
      f"layers={config_karatsuba['model']['n_shared_layers']}")
print(f"Algorithm: Karatsuba recursive decomposition")
print(f"Loops: {config_karatsuba['model']['loop']['n_loops_start']} -> {config_karatsuba['model']['loop']['n_loops_max']}")
print(f"Training steps: {config_karatsuba['training']['n_steps']}")
print(f"Curriculum: {[s['name'] for s in config_karatsuba['data']['curriculum']['stages']]}")
print()

wandb.init(
    project=config_karatsuba['experiment']['wandb_project'],
    name=config_karatsuba['experiment']['name'],
    tags=config_karatsuba['experiment']['wandb_tags'],
    config=config_karatsuba,
)

# Build dataset with Karatsuba traces
dataset_karatsuba = MultiplicationDataset(config_karatsuba['data'])
print(f"Dataset size: {len(dataset_karatsuba)} examples")

# Build model
key = jrandom.PRNGKey(config_karatsuba['experiment']['seed'])
model_karatsuba = KaratsubaTransformer(config_karatsuba['model'], key=key)
n_params = sum(x.size for x in jtu.tree_leaves(model_karatsuba) if hasattr(x, 'size'))
print(f"Model parameters: {n_params:,}")

# Train with curriculum
trainer_karatsuba = Trainer(config_karatsuba['training'], model_karatsuba, dataset_karatsuba)

try:
    model_karatsuba, history_karatsuba = trainer_karatsuba.train()
except KeyboardInterrupt:
    print("\nTraining interrupted. Saving checkpoint...")
    trainer_karatsuba.save_checkpoint()
    history_karatsuba = trainer_karatsuba.get_history()

print(f"\nFinal 8-bit exact-match accuracy: {history_karatsuba['eval_exact_match'][-1]:.4f}")
wandb.finish()

In [ ]:
# Cell 5: Compare training curves

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Training Loss ---
ax = axes[0]
if 'train_loss' in history_school:
    steps_s = np.arange(len(history_school['train_loss'])) * config_school['training']['log_every']
    ax.plot(steps_s, history_school['train_loss'], label='School', alpha=0.8, color='red')
if 'train_loss' in history_karatsuba:
    steps_k = np.arange(len(history_karatsuba['train_loss'])) * config_karatsuba['training']['log_every']
    ax.plot(steps_k, history_karatsuba['train_loss'], label='Karatsuba', alpha=0.8, color='blue')
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training Loss', fontsize=14)
ax.legend(fontsize=11)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# --- Training Accuracy (8-bit) ---
ax = axes[1]
if 'eval_exact_match' in history_school:
    steps_s = np.arange(len(history_school['eval_exact_match'])) * config_school['training']['eval_every']
    ax.plot(steps_s, history_school['eval_exact_match'], 'ro-', label='School', markersize=4)
if 'eval_exact_match' in history_karatsuba:
    steps_k = np.arange(len(history_karatsuba['eval_exact_match'])) * config_karatsuba['training']['eval_every']
    ax.plot(steps_k, history_karatsuba['eval_exact_match'], 'bs-', label='Karatsuba', markersize=4)
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Exact Match Accuracy', fontsize=12)
ax.set_title('8-bit Eval Accuracy', fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)

# --- Per-digit accuracy at final step ---
ax = axes[2]
if 'per_digit_accuracy' in history_school:
    digits = range(len(history_school['per_digit_accuracy'][-1]))
    ax.bar([d - 0.15 for d in digits], history_school['per_digit_accuracy'][-1],
           width=0.3, label='School', color='red', alpha=0.7)
if 'per_digit_accuracy' in history_karatsuba:
    digits = range(len(history_karatsuba['per_digit_accuracy'][-1]))
    ax.bar([d + 0.15 for d in digits], history_karatsuba['per_digit_accuracy'][-1],
           width=0.3, label='Karatsuba', color='blue', alpha=0.7)
ax.set_xlabel('Output Bit Position (0=LSB)', fontsize=12)
ax.set_ylabel('Per-Bit Accuracy', fontsize=12)
ax.set_title('Per-Bit Accuracy (8-bit, Final)', fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_BASE, 'training_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nSummary:")
print(f"  School 8-bit accuracy:    {history_school.get('eval_exact_match', [0])[-1]:.4f}")
print(f"  Karatsuba 8-bit accuracy: {history_karatsuba.get('eval_exact_match', [0])[-1]:.4f}")